# 02. Baseline Retrieval — Qualitative Check

Before quantitative sweeps, we eyeball retrieval on a small set of probe
queries to confirm:

1. The collection is reachable and returns sensible hits.
2. The cosine-score range we'll be sweeping over is realistic.
3. An adversarial / out-of-domain query returns nothing (or very low scores).

This notebook deliberately uses **`retrieve_simple`** (pure cosine,
no query expansion, no severity / applicability boosts) so we observe
the raw embedding signal — the boosts are studied separately in notebook 05.


In [1]:
import os, sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

os.environ["QDRANT_URL"] = "http://localhost:6433"
os.environ["AGENT_KMS_COLLECTION"] = "eval_corpus"
os.environ["AGENT_KMS_CONFIG"] = str(ROOT / ".agent-kms" / "kms.toml")
os.environ["AGENT_KMS_KNOWLEDGE_ROOT"] = str(ROOT / "data" / "corpus")
sys.path.insert(0, str(ROOT))

from agent_kms.retrieve import retrieve_simple


/Users/toshihiro-yamaguchi/dev/portfolio/rag-evaluation-jp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
probe_queries = [
    # in-domain, direct
    "useStateでstate変数を追加する",
    # in-domain, paraphrased
    "親から渡された値を子コンポーネントで初期値に使ったあと書き換えたい",
    # JS, technical
    "try catch で例外を捕捉する書き方",
    # JS, intent
    "配列を変更せずに新しい配列を返したい",
    # adversarial — out-of-domain
    "PostgreSQL の logical replication を設定する手順",
]

for q in probe_queries:
    print(f"\n──── {q!r}")
    hits = retrieve_simple(q, limit=5)
    for h in hits:
        src = h["source"].split("/")[-1]
        print(f"  {h['score']:.3f}  {src:45s}  | {h['heading'][:50]}")



──── 'useStateでstate変数を追加する'


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9804.50it/s]

  0.869  react-state-a-components-memory.md             | state 変数の追加 {/*adding-a-state-variable*/}
  0.867  react-state-a-components-memory.md             | (preamble)
  0.864  react-state-a-components-memory.md             | コンポーネントで複数の state 変数を使う {/*giving-a-component-mult
  0.858  react-managing-state.md                        | state 構造の選択 {/*choosing-the-state-structure*/}
  0.852  react-managing-state.md                        | (preamble)

──── '親から渡された値を子コンポーネントで初期値に使ったあと書き換えたい'
  0.845  react-passing-props-to-a-component.md          | (preamble)
  0.841  jsprimer-object.md                             | プロパティへのアクセス {#property-access}
  0.837  react-passing-props-to-a-component.md          | コンポーネントに props を渡す {/*passing-props-to-a-component
  0.836  jsprimer-class.md                              | 継承 {#extends}
  0.835  jsprimer-object.md                             | プロパティの追加 {#add-property}

──── 'try catch で例外を捕捉する書き方'
  0.887  jsprimer-error-try-catch.md                  

/Users/toshihiro-yamaguchi/dev/portfolio/rag-evaluation-jp/.venv/lib/python3.12/site-packages/qdrant_client/qdrant_remote.py:282: UserWarning: Qdrant client version 1.18.0 is incompatible with server version 1.12.0. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


## Observations

- **In-domain queries top out around cosine ≈ 0.87** for `multilingual-e5-base`.
  This is well **below the agent-kms default threshold of 0.93** — on this
  corpus, the default would silently return zero results for most queries.
- The Japanese-on-Japanese baseline cosine is high (~0.80 floor) even for
  unrelated content, so the *discrimination* between relevant and noise
  is in a narrow 0.05-0.08 band, not a 0.10+ band.
- The out-of-domain query (PostgreSQL) returns hits at ~0.78-0.80 — i.e.
  the model **does** confidently assign lower scores to irrelevant content,
  but that lower band overlaps with the on-topic band of harder paraphrased
  queries. The threshold sweep in notebook 03 quantifies this trade-off.
